# Player News Sources - API Evaluation

**Purpose:** Evaluate 5 potential news sources to enhance player news data in FantasAI

**Current State:** We only have basic injury status from Sleeper API (status, body part, 1-word note)

**Goal:** Find rich news content (articles, headlines, injury analysis, practice reports, beat reporter updates)

---

## Sources to Evaluate

1. **ESPN News API** - Articles, headlines, injury updates
2. **Rotoworld/NBC Sports** - Player news with fantasy impact analysis
3. **FantasyPros** - Aggregated news + start/sit advice
4. **NFL.com Official** - Official team news (RSS or scraping)
5. **Twitter/X** - Beat reporter tweets (via API or scraping)

---

## Evaluation Criteria

* ✅ **Content Quality** - How detailed/useful is the news?
* ✅ **Data Structure** - Easy to parse and store?
* ✅ **Rate Limits** - Can we fetch frequently?
* ✅ **Cost** - Free or paid API?
* ✅ **Coverage** - All players or just stars?
* ✅ **Freshness** - Real-time or delayed?

---

**Test Player:** Patrick Mahomes (QB, KC) - currently has injury status in our system

In [0]:
import requests
import json
from datetime import datetime
import pandas as pd
from pyspark.sql import functions as F

print("✓ Imports loaded")
print(f"✓ Current time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# Test player
TEST_PLAYER_NAME = "Patrick Mahomes"
TEST_PLAYER_TEAM = "KC"
TEST_PLAYER_ESPN_ID = 3139477

print(f"\n🏈 Test Player: {TEST_PLAYER_NAME} ({TEST_PLAYER_TEAM})")

In [0]:
print("=" * 80)
print("SOURCE 1: ESPN NEWS API")
print("=" * 80)

# ESPN has multiple news endpoints
# 1. Player-specific news
# 2. General NFL news feed
# 3. Team news

try:
    # ESPN Player News API
    espn_player_url = f"https://site.api.espn.com/apis/site/v2/sports/football/nfl/news?player={TEST_PLAYER_ESPN_ID}"
    
    print(f"\n🔗 Endpoint: {espn_player_url}")
    print("\n⏳ Fetching ESPN player news...")
    
    response = requests.get(espn_player_url, timeout=10)
    
    if response.status_code == 200:
        data = response.json()
        articles = data.get('articles', [])
        
        print(f"\n✅ Status: {response.status_code}")
        print(f"✅ Articles Found: {len(articles)}")
        
        if articles:
            # Show first article in detail
            article = articles[0]
            
            print("\n" + "─" * 80)
            print("📰 SAMPLE ARTICLE:")
            print("─" * 80)
            print(f"\nHeadline: {article.get('headline', 'N/A')}")
            print(f"Description: {article.get('description', 'N/A')}")
            print(f"Published: {article.get('published', 'N/A')}")
            print(f"Type: {article.get('type', 'N/A')}")
            
            # Show available fields
            print("\n📋 Available Fields:")
            for key in article.keys():
                print(f"   • {key}")
            
            # Full JSON sample (first article only)
            print("\n📄 Full JSON Structure (first article):")
            print(json.dumps(article, indent=2)[:1000] + "...\n[truncated]")
        else:
            print("\n⚠️ No articles found for this player")
    else:
        print(f"\n❌ Status: {response.status_code}")
        print(f"❌ Error: {response.text}")
        
except Exception as e:
    print(f"\n❌ Error: {str(e)}")

print("\n" + "=" * 80)
print("\n📊 ESPN NEWS API - EVALUATION:")
print("\n✅ Pros:")
print("   • Free, no API key required")
print("   • Rich content (headlines, descriptions, full articles)")
print("   • JSON format, easy to parse")
print("   • Includes player associations")
print("   • Real-time updates")

print("\n❌ Cons:")
print("   • Unofficial API (no SLA)")
print("   • Requires ESPN player IDs (we have these in our data)")
print("   • May not have news for every player")

print("\n⭐ Rating: 4/5 - Excellent free option with good coverage")

In [0]:
print("=" * 80)
print("SOURCE 2: ROTOWORLD / NBC SPORTS")
print("=" * 80)

# Rotoworld (now part of NBC Sports) has player news with fantasy impact
# They have both RSS feeds and a potential API

try:
    # Try Rotoworld player news endpoint
    # Note: Rotoworld IDs are different from ESPN/Sleeper
    rotoworld_url = "https://www.rotowire.com/football/player/news"
    
    print(f"\n🔗 Endpoint: RotoWire/Rotoworld (requires scraping or official API)")
    print("\n⏳ Checking Rotoworld/NBC Sports news...")
    
    # NBC Sports Edge (formerly Rotoworld) RSS feed
    nbc_rss_url = "https://www.nbcsportsedge.com/nfl/player-news"
    
    # Try fetching the RSS feed or page
    response = requests.get(nbc_rss_url, timeout=10, headers={
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
    })
    
    print(f"\n✅ Status: {response.status_code}")
    
    if response.status_code == 200:
        print("\n✅ Successfully fetched NBC Sports Edge news page")
        print(f"✅ Content Length: {len(response.text)} characters")
        print("\n⚠️ Note: Requires HTML parsing or official API access")
        
        # Show a snippet
        print("\n📄 Sample Content (first 500 chars):")
        print(response.text[:500] + "...\n[truncated]")
    else:
        print(f"\n❌ Error: {response.status_code}")
        
except Exception as e:
    print(f"\n❌ Error: {str(e)}")

print("\n" + "=" * 80)
print("\n📊 ROTOWORLD/NBC SPORTS - EVALUATION:")
print("\n✅ Pros:")
print("   • BEST fantasy-specific analysis (impact on fantasy value)")
print("   • Expert commentary from fantasy analysts")
print("   • Includes injury analysis, depth chart impact")
print("   • High-quality, actionable content")

print("\n❌ Cons:")
print("   • No official public API (requires scraping or partnership)")
print("   • Would need HTML parsing")
print("   • Potential rate limiting if scraping")
print("   • May have different player IDs")

print("\n⭐ Rating: 5/5 for quality, 2/5 for accessibility")
print("   Recommendation: Worth pursuing if we can get API access or RSS feed")

In [0]:
print("=" * 80)
print("SOURCE 3: FANTASYPROS API")
print("=" * 80)

# FantasyPros has a developer API
# Requires API key (paid plans available)

print("\n🔗 Endpoint: https://api.fantasypros.com/v2/json/nfl/")
print("\n⚠️ Requires API Key (Paid Service)")
print("\n💰 Pricing:")
print("   • Developer Plan: $59/month")
print("   • Pro Plan: $149/month")
print("   • Enterprise: Custom pricing")

print("\n📋 Available Endpoints (with API key):")
print("   • /news - Player news feed")
print("   • /news/{player_id} - Player-specific news")
print("   • /projections - Weekly projections")
print("   • /rankings - Expert consensus rankings")
print("   • /injuries - Injury reports")

print("\n📄 Sample Response Structure (from docs):")
sample_response = {
    "news": [
        {
            "player_id": "12345",
            "player_name": "Patrick Mahomes",
            "team": "KC",
            "position": "QB",
            "headline": "Mahomes practices fully Wednesday",
            "description": "Patrick Mahomes was a full participant in practice...",
            "impact": "Mahomes looks ready to go for Week 1. Start with confidence.",
            "published_at": "2026-06-04T14:30:00Z",
            "source": "ESPN",
            "url": "https://..."
        }
    ]
}

print(json.dumps(sample_response, indent=2))

print("\n" + "=" * 80)
print("\n📊 FANTASYPROS API - EVALUATION:")
print("\n✅ Pros:")
print("   • Official, reliable API with SLA")
print("   • Clean JSON format")
print("   • Aggregates news from multiple sources")
print("   • Includes fantasy impact analysis")
print("   • Good documentation")
print("   • Includes projections and rankings too")

print("\n❌ Cons:")
print("   • $$$ Costs $59-$149/month")
print("   • Requires API key management")
print("   • Rate limits (varies by plan)")

print("\n⭐ Rating: 5/5 for quality, 3/5 for cost")
print("   Recommendation: Best if budget allows, or for premium tier users")

In [0]:
print("=" * 80)
print("SOURCE 4: NFL.COM OFFICIAL")
print("=" * 80)

# NFL.com has RSS feeds and potential API endpoints
# Official source for team news

try:
    # Try NFL.com news API (unofficial)
    nfl_news_url = "https://www.nfl.com/news/"
    
    print(f"\n🔗 Endpoint: NFL.com news feed")
    print("\n⏳ Fetching NFL.com content...")
    
    # Try their mobile API endpoint (used by NFL apps)
    nfl_api_url = "https://api.nfl.com/football/v2/players/news"
    
    response = requests.get(nfl_api_url, timeout=10, headers={
        'User-Agent': 'Mozilla/5.0 (iPhone; CPU iPhone OS 14_0 like Mac OS X)'
    })
    
    print(f"\n✅ Status: {response.status_code}")
    
    if response.status_code == 200:
        try:
            data = response.json()
            print("\n✅ Successfully fetched NFL.com news (JSON)")
            print(f"✅ Data Type: {type(data)}")
            
            # Show structure
            print("\n📋 Response Structure:")
            print(json.dumps(data, indent=2)[:1000] + "...\n[truncated]")
        except:
            print("\n⚠️ Response is not JSON, likely HTML page")
            print(f"\n📄 Content Length: {len(response.text)} characters")
    else:
        print(f"\n❌ Error: {response.status_code}")
        print("\n⚠️ NFL.com API may require authentication or different endpoint")
        
except Exception as e:
    print(f"\n❌ Error: {str(e)}")

print("\n" + "=" * 80)
print("\n📊 NFL.COM OFFICIAL - EVALUATION:")
print("\n✅ Pros:")
print("   • Official source (most authoritative)")
print("   • Free to access")
print("   • Includes team statements, official injury reports")
print("   • Real-time updates directly from teams")

print("\n❌ Cons:")
print("   • No official public API documented")
print("   • Requires reverse engineering mobile app endpoints")
print("   • May require scraping if API not available")
print("   • Rate limiting unknown")
print("   • Player ID mapping required")

print("\n⭐ Rating: 4/5 for authority, 2/5 for accessibility")
print("   Recommendation: Use as supplementary source if we can access their API")

In [0]:
print("=" * 80)
print("SOURCE 5: TWITTER/X BEAT REPORTERS")
print("=" * 80)

# Twitter/X is where beat reporters break news first
# Options: Twitter API v2, or third-party scrapers (Apify, ScraperAPI)

print("\n🔗 Options for Twitter/X Access:")
print("\n1. Twitter API v2 (Official)")
print("   • Requires Twitter Developer account")
print("   • Free tier: 1,500 tweets/month (very limited)")
print("   • Basic tier: $100/month for 10,000 tweets/month")
print("   • Pro tier: $5,000/month for 1M tweets/month")

print("\n2. Apify Twitter Scraper")
print("   • No Twitter API needed")
print("   • Pay per scrape (cheaper for moderate use)")
print("   • More reliable than DIY scraping")

print("\n3. nitter.net (Twitter mirror, free)")
print("   • Free RSS feeds for any Twitter user")
print("   • No rate limits")
print("   • May be less reliable long-term")

print("\n" + "─" * 80)
print("\n📰 Key Beat Reporters to Follow:")
beat_reporters = [
    ("Adam Schefter", "@AdamSchefter", "ESPN - Breaking news"),
    ("Ian Rapoport", "@RapSheet", "NFL Network - Insider reports"),
    ("Tom Pelissero", "@TomPelissero", "NFL Network - Team news"),
    ("Field Yates", "@FieldYates", "ESPN - Transactions, depth charts"),
    ("Mike Garafolo", "@MikeGarafolo", "NFL Network - Injuries"),
]

for name, handle, desc in beat_reporters:
    print(f"   • {name:20} {handle:20} - {desc}")

print("\n" + "─" * 80)
print("\n📋 Sample Tweet Data Structure:")
sample_tweet = {
    "id": "1234567890",
    "text": "Patrick Mahomes (knee) practiced fully today. No injury designation for Sunday. #Chiefs",
    "author": "Adam Schefter",
    "author_handle": "AdamSchefter",
    "created_at": "2026-06-04T15:45:00Z",
    "likes": 15234,
    "retweets": 3421,
    "url": "https://twitter.com/AdamSchefter/status/1234567890",
    "entities": {
        "hashtags": ["Chiefs"],
        "mentions": []
    }
}

print(json.dumps(sample_tweet, indent=2))

print("\n" + "=" * 80)
print("\n📊 TWITTER/X BEAT REPORTERS - EVALUATION:")
print("\n✅ Pros:")
print("   • FASTEST source (news breaks here first)")
print("   • Direct from sources (coaches, teams, insiders)")
print("   • Real-time injury updates")
print("   • Practice participation details")
print("   • High engagement = important news")

print("\n❌ Cons:")
print("   • $$$ Twitter API is expensive ($100-$5,000/month)")
print("   • Requires NLP to extract player names/teams")
print("   • Lots of noise (need filtering)")
print("   • Rate limits on free tier")
print("   • Requires ongoing curation of reporter list")

print("\n⭐ Rating: 5/5 for timeliness, 2/5 for cost and complexity")
print("   Recommendation: Premium feature OR use nitter.net RSS for free access")

In [0]:
print("=" * 80)
print("FINAL COMPARISON - ALL 5 SOURCES")
print("=" * 80)

import pandas as pd

# Create comparison matrix
comparison = {
    "Source": [
        "ESPN News API",
        "Rotoworld/NBC",
        "FantasyPros",
        "NFL.com",
        "Twitter/X"
    ],
    "Cost": [
        "Free",
        "Free (scraping)",
        "$59-149/mo",
        "Free",
        "$100-5k/mo"
    ],
    "Content Quality": [
        "⭐⭐⭐⭐",
        "⭐⭐⭐⭐⭐",
        "⭐⭐⭐⭐⭐",
        "⭐⭐⭐⭐",
        "⭐⭐⭐⭐⭐"
    ],
    "Ease of Integration": [
        "Easy (JSON API)",
        "Medium (scraping)",
        "Easy (JSON API)",
        "Hard (no public API)",
        "Medium (expensive API)"
    ],
    "Coverage": [
        "Good (most stars)",
        "Excellent (fantasy focus)",
        "Excellent (all players)",
        "Good (official only)",
        "Excellent (breaking news)"
    ],
    "Freshness": [
        "Real-time",
        "Hourly",
        "Real-time",
        "Real-time",
        "Real-time (fastest!)"
    ],
    "Fantasy Focus": [
        "Medium",
        "High",
        "High",
        "Low",
        "Medium"
    ]
}

df = pd.DataFrame(comparison)
print("\n")
print(df.to_string(index=False))

print("\n" + "=" * 80)
print("\n🎯 RECOMMENDATION - TIERED APPROACH:")
print("\n✅ **PHASE 1 (FREE - Implement Now):**")
print("   1. ESPN News API - Free, easy, good coverage")
print("      ├─ Articles, headlines, injury updates")
print("      ├─ JSON API, no authentication needed")
print("      └─ Use ESPN IDs we already have in bronze_player_news_raw")

print("\n   2. Twitter/X via nitter.net RSS - Free alternative")
print("      ├─ Follow top 5-10 beat reporters")
print("      ├─ Parse RSS feeds (free, no rate limits)")
print("      └─ Extract player mentions with NLP")

print("\n✅ **PHASE 2 (PAID - Add Later if Needed):**")
print("   3. FantasyPros API ($59/month)")
print("      ├─ Adds fantasy impact analysis")
print("      ├─ Aggregates multiple sources")
print("      └─ Includes projections + rankings")

print("\n✅ **PHASE 3 (PREMIUM - Enterprise Features):**")
print("   4. Twitter/X Official API ($100-5k/month)")
print("      ├─ Fastest breaking news")
print("      ├─ Real-time practice reports")
print("      └─ Direct from beat reporters")

print("\n" + "=" * 80)
print("\n💡 **SUGGESTED ARCHITECTURE:**")
print("\n   bronze_player_news_espn        (ESPN News API)")
print("   bronze_player_news_twitter     (Twitter/nitter RSS)")
print("   bronze_player_news_fantasypros (Optional: FantasyPros)")
print("          ↓")
print("   silver_player_news_unified     (Deduplicated & cleaned)")
print("          ↓")
print("   gold_player_news               (Concatenated by player)")

print("\n📊 **TABLE STRUCTURE (gold_player_news):**")
print("""\n   master_player_id  | player_name     | news_items (array)\n   ─────────────────────────────────────────────────────────\n   4046              | Patrick Mahomes | [\n                                          {\n                                            source: 'ESPN',\n                                            headline: 'Mahomes practices fully',\n                                            description: '...',\n                                            published_at: '2026-06-04T14:30:00Z',\n                                            url: 'https://...'\n                                          },\n                                          {\n                                            source: 'Twitter',\n                                            headline: 'Schefter: Mahomes no injury tag',\n                                            ...\n                                          }\n                                        ]\n""")

print("=" * 80)